In [0]:
from pyspark.sql.functions import col, current_timestamp
import requests
import pandas as pd
import io
import re

spark.sql("USE CATALOG scottish_housing_ws")
spark.sql("USE SCHEMA bronze")

In [0]:
url = "https://statistics.gov.scot/downloads/cube-table?uri=http://statistics.gov.scot/data/new-build-housing-starts-and-completions"

response = requests.get(url)
response.raise_for_status()

df_raw = pd.read_csv(io.StringIO(response.text))

print(df_raw.shape)
print(df_raw.dtypes)
df_raw.head(10)

In [0]:
def clean_col_name(name):
    name = name.lower()
    name = re.sub(r"[^\w]", "_", name)   # replace non-word chars with underscore
    name = re.sub(r"_+", "_", name)       # collapse multiple underscores
    name = name.strip("_")                # strip leading/trailing underscores
    return name

df_raw.columns = [clean_col_name(c) for c in df_raw.columns]

print(df_raw.columns.tolist())
df_raw.head(5)

In [0]:
df_bronze = spark.createDataFrame(df_raw)
df_bronze.printSchema()
df_bronze.select("*").show(10, truncate=False)

In [0]:
(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("scottish_housing_ws.bronze.housebuilding")
)

In [0]:
# Source: statistics.gov.scot - New Build Housing Starts and Completions
# URL: https://statistics.gov.scot/downloads/cube-table?uri=http://statistics.gov.scot/data/new-build-housing-starts-and-completions
# Column names sanitised in pandas before Spark ingestion to avoid Delta special character rejection.
# All rows and columns ingested as-is. Filtering and shaping happens in silver.
# Open Government Licence.